# 04 — Credit Scoring & Probability of Default Estimation

## Purpose
Combine financial ratios, Altman Z-score, and Merton structural model into an integrated credit scorecard that produces an internal rating and committee decision.

**Bank context:** Every commercial borrower gets an internal credit rating based on quantitative metrics. This rating drives:
- Approval/decline decisions
- Pricing (risk-based margin)
- Regulatory capital (RWA under IRB)
- Portfolio monitoring and watchlist triggers

**This notebook maps to three Excel sheets:**
- `Sigma_Final` — Altman Z-Score with SME proxy
- `Merton_PD_EL` — Structural PD and Expected Loss
- `IC_Decision` — Integrated Credit Scorecard

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generation import generate_sme_dataset
from src.ratio_engine import calculate_ratios
from src.altman_zscore import borrower_zscore_trend, portfolio_zscore_summary
from src.merton_pd import borrower_merton_analysis, portfolio_merton_summary
from src.credit_scorecard import score_borrower, portfolio_scorecard_summary

pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

df = generate_sme_dataset(n_borrowers=80, seed=42)
df_r = calculate_ratios(df)

## Part 1: Altman Z-Score (SME Proxy)

The Altman Z-score uses 5 financial ratios to estimate default probability. Since SMEs are not listed, we use **book equity** as a proxy for market capitalisation (the Z' model approach).

**Formula:** Z = 1.2 * T1 + 1.4 * T2 + 3.3 * T3 + 0.6 * T4 + 1.0 * T5

| Component | Formula | Meaning |
|-----------|---------|--------|
| T1 | Working Capital / Total Assets | Liquidity |
| T2 | Retained Earnings / Total Assets | Cumulative profitability |
| T3 | EBIT / Total Assets | Operating efficiency |
| T4 | Book Equity / Total Liabilities | Solvency (market cap proxy) |
| T5 | Sales / Total Assets | Asset turnover |

In [ ]:
# Z-score trend for base case borrower
zt = borrower_zscore_trend(df, borrower_id=0)

print('='*70)
print('ALTMAN Z-SCORE TREND — Base Case AU SME Pty Ltd')
print(f'Trend: {zt.attrs.get("trend", "N/A")} — {zt.attrs.get("trend_comment", "")}')
print('='*70)
display(zt)

print(f'\nFY0 Z-score: {zt.loc["FY0", "zscore"]} → {zt.loc["FY0", "zone"]} zone')
print(f'(Verification: Excel model = 2.885 → GREY zone)')

In [ ]:
# Z-score trend chart
fig, ax = plt.subplots(figsize=(10, 6))
periods = ['FY-2', 'FY-1', 'FY0']
scores = zt['zscore'].values

ax.plot(periods, scores, 'o-', linewidth=3, markersize=10, color='#2196F3')

# Zone bands
ax.axhspan(2.99, max(scores) + 1, alpha=0.1, color='green', label='Safe Zone (>2.99)')
ax.axhspan(1.8, 2.99, alpha=0.1, color='orange', label='Grey Zone (1.8-2.99)')
ax.axhspan(0, 1.8, alpha=0.1, color='red', label='Distress Zone (<1.8)')

ax.axhline(2.99, color='green', linestyle='--', alpha=0.5)
ax.axhline(1.8, color='red', linestyle='--', alpha=0.5)

for i, s in enumerate(scores):
    ax.annotate(f'{s:.3f}', (periods[i], s), textcoords='offset points',
                xytext=(0, 15), ha='center', fontsize=12, fontweight='bold')

ax.set_title('Altman Z-Score Trend (SME Proxy)', fontsize=14, fontweight='bold')
ax.set_ylabel('Z-Score')
ax.legend(loc='lower right')
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('../outputs/figures/04_zscore_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 2: Merton Structural PD Model

The Merton model treats firm equity as a **call option** on assets. Default occurs when asset value falls below the debt barrier.

**Key outputs:**
- **Distance-to-Default (DD):** How many standard deviations the firm is from default
- **PD = N(-DD):** Probability of default over 1 year
- **PVEL:** Present Value of Expected Loss (pricing input)

In [ ]:
merton = borrower_merton_analysis(df, borrower_id=0)

print('='*70)
print('MERTON PD MODEL — Base Case AU SME Pty Ltd')
print('='*70)

merton_display = pd.DataFrame([
    ('Asset value (A)', f'${merton["asset_value"]:,.0f}', 'Proxy = Total Assets'),
    ('Debt threshold (D)', f'${merton["debt_threshold"]:,.0f}', 'Proxy = Total Debt'),
    ('Expected return (mu)', f'{merton["mu"]:.4f}', 'Industry proxy'),
    ('Volatility (sigma)', f'{merton["sigma"]:.4f}', 'Sector proxy from Sigma_Final'),
    ('Risk-free rate (r)', f'{merton["rf"]:.4f}', 'AU proxy'),
    ('Distance-to-Default', f'{merton["dd"]:.4f}', 'Higher = further from default'),
    ('PD = N(-DD)', f'{merton["pd"]:.4%}', merton['pd_comment']),
    ('PVEL', f'${merton["pvel"]:,.0f}', merton['pvel_comment']),
    ('Implied LGD', f'{merton["implied_lgd"]:.2%}', 'PVEL / discounted default leg'),
    ('Leverage', f'{merton["debt_threshold"]/merton["asset_value"]:.1%}', merton['leverage_comment']),
], columns=['Metric', 'Value', 'Comment'])

display(merton_display)
print(f'\n(Verification: Excel PD = 5.36%, PVEL = $62,578)')

## Part 3: Integrated Credit Scorecard

The scorecard combines all metrics into a single weighted score and committee decision.

**Grading:** A (>=80%) → APPROVE | B (>=60%) → APPROVE | C (>=40%) → REFER | D (<40%) → DECLINE

In [ ]:
sc = score_borrower(df_r, borrower_id=0)

print('='*70)
print('INTEGRATED CREDIT SCORECARD — Base Case AU SME Pty Ltd')
print(f'Weighted Score: {sc["weighted_score"]:.2%} | Grade: {sc["grade"]} | Decision: {sc["decision"]}')
print(f'Tests Passed: {sc["total_passed"]} / {sc["total_tests"]}')
print('='*70)

# Format scorecard table
sc_table = sc['scorecard_table'].copy()
sc_table['actual_display'] = sc_table['actual'].apply(
    lambda x: f'{x:.4f}' if isinstance(x, (int, float)) and not pd.isna(x) else str(x)
)

def colour_pass(val):
    return 'background-color: #C8E6C9' if val == 1 else 'background-color: #FFCDD2'

display(
    sc_table[['metric', 'actual_display', 'rule', 'pass', 'weight']]
    .rename(columns={'actual_display': 'actual'})
    .style.applymap(colour_pass, subset=['pass'])
)

print(f'\nNarrative: {sc["narrative"]}')

## Portfolio Distributions

In [ ]:
# Portfolio-level summaries
port_zs = portfolio_zscore_summary(df)
port_pd = portfolio_merton_summary(df)
port_sc = portfolio_scorecard_summary(df_r)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Z-score distribution
zone_colors = port_zs['zone'].map({'SAFE': '#4CAF50', 'GREY': '#FF9800', 'DISTRESS': '#F44336'})
axes[0, 0].scatter(range(len(port_zs)), port_zs['zscore'].sort_values(ascending=False),
                    c=zone_colors.iloc[port_zs['zscore'].sort_values(ascending=False).index], s=30, alpha=0.7)
axes[0, 0].axhline(2.99, color='green', linestyle='--', alpha=0.7, label='Safe >2.99')
axes[0, 0].axhline(1.8, color='red', linestyle='--', alpha=0.7, label='Distress <1.8')
axes[0, 0].set_title('Z-Score Distribution', fontweight='bold')
axes[0, 0].set_ylabel('Z-Score')
axes[0, 0].legend()

# PD distribution
axes[0, 1].hist(port_pd['pd'] * 100, bins=20, color='#2196F3', alpha=0.8, edgecolor='white')
axes[0, 1].set_title('Merton PD Distribution', fontweight='bold')
axes[0, 1].set_xlabel('PD (%)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(10, color='red', linestyle='--', label='10% threshold')
axes[0, 1].legend()

# Grade distribution
grade_counts = port_sc['grade'].value_counts().reindex(['A', 'B', 'C', 'D'], fill_value=0)
grade_colors = {'A': '#4CAF50', 'B': '#8BC34A', 'C': '#FF9800', 'D': '#F44336'}
axes[1, 0].bar(grade_counts.index, grade_counts.values,
               color=[grade_colors[g] for g in grade_counts.index], alpha=0.8)
axes[1, 0].set_title('Internal Grade Distribution', fontweight='bold')
axes[1, 0].set_ylabel('Number of Borrowers')

# Decision distribution
decision_counts = port_sc['decision'].value_counts()
decision_colors = {'APPROVE': '#4CAF50', 'REFER': '#FF9800', 'DECLINE': '#F44336'}
axes[1, 1].pie(decision_counts.values,
               labels=[f'{k}\n({v})' for k, v in decision_counts.items()],
               colors=[decision_colors.get(k, '#9E9E9E') for k in decision_counts.index],
               autopct='%1.0f%%', startangle=90, textprops={'fontsize': 11})
axes[1, 1].set_title('Committee Decision Distribution', fontweight='bold')

fig.suptitle('Portfolio Credit Quality Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/04_portfolio_credit_quality.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Z-score zones: {port_zs["zone"].value_counts().to_dict()}')
print(f'Median PD: {port_pd["pd"].median():.2%}')
print(f'Grades: {port_sc["grade"].value_counts().to_dict()}')

---

## Key Takeaways

1. **Z-score provides a quick health check** — the 3-period trend is as important as the absolute level
2. **Merton PD connects balance sheet leverage to default probability** — key for IRB capital calculations
3. **The integrated scorecard combines multiple views** — no single metric drives the decision
4. **Grade distribution shows portfolio quality** — concentration in A/B grades is preferred

**Next:** [05_Risk_Based_Pricing.ipynb](05_Risk_Based_Pricing.ipynb)